In [8]:
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications.efficientnet import preprocess_input
from sklearn.utils.class_weight import compute_class_weight

# Mount Google Drive contents
from google.colab import drive
drive.mount('/content/drive')

dataset_path = "/content/drive/MyDrive/Dataset"

# Padding and Resizeing Function
def pad_and_resize(img, target_size=(224, 224)):
    h, w = img.shape[:2]
    max_dim = max(h, w)
    top = (max_dim - h) // 2
    bottom = max_dim - h - top
    left = (max_dim - w) // 2
    right = max_dim - w - left
    img_padded = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=[255, 255, 255])
    img_resized = cv2.resize(img_padded, target_size)
    return preprocess_input(img_resized.astype(np.float32))  # ✅ Normalize using preprocess_input

# Data Augmentation
aug_gen = ImageDataGenerator(
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

# Load images
X, y, labels = [], [], []
class_names = sorted(os.listdir(dataset_path))

for class_index, class_name in enumerate(class_names):
    class_path = os.path.join(dataset_path, class_name)
    if os.path.isdir(class_path):
        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = pad_and_resize(img)
                X.append(img)
                y.append(class_index)
                labels.append(class_name)

X = np.array(X, dtype=np.float32)
y = np.array(y)
y = to_categorical(y, num_classes=len(class_names))

# Compute Class Weights (mainly for Imbalanced Data)
class_weights = compute_class_weight("balanced", classes=np.unique(y.argmax(axis=1)), y=y.argmax(axis=1))
class_weights = {i: weight for i, weight in enumerate(class_weights)}

# Augment Dataset (Limited Augmentations Per Image)
augmented_images, augmented_labels = [], []
for i in range(len(X)):
    img = X[i].reshape((1,) + X[i].shape)
    label = y[i]
    count = 0
    for batch in aug_gen.flow(img, batch_size=1):
        augmented_images.append(batch[0])
        augmented_labels.append(label)
        count += 1
        if count >= 3:  # Increased to 3 augmented images per original image
            break

X_aug = np.array(augmented_images, dtype=np.float32)
y_aug = np.array(augmented_labels)

# Combining Original and Augmented Data
X_combined = np.concatenate((X, X_aug), axis=0)
y_combined = np.concatenate((y, y_aug), axis=0)

# Train-Test Split (20% Test)
X_train, X_test, y_train, y_test = train_test_split(X_combined, y_combined, test_size=0.2, random_state=42)

# Load EfficientNetB0
base_model = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# Freeze Pretrained Layers Initially
for layer in base_model.layers:
    layer.trainable = False

# Model Architecture
x = GlobalAveragePooling2D()(base_model.output)
x = BatchNormalization()(x)  # Add Batch Normalization
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)  #  Add Dropout
output = Dense(len(class_names), activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer=Adam(learning_rate=1e-4), loss="categorical_crossentropy", metrics=["accuracy"])

#  Callbacks for Learning Rate & Early Stopping
callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1),
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)
]

# Train Model
history = model.fit(
    X_train, y_train,
    epochs=25,
    batch_size=32,
    validation_data=(X_test, y_test),
    class_weight=class_weights,  #  Helps with class imbalance
    callbacks=callbacks
)

# Unfreeze Some Layers for Fine-tuning (After Initial Training)
for layer in base_model.layers[-20:]:  #  Unfreeze last 20 layers
    layer.trainable = True

#  Recompile with Lower Learning Rate for Fine-tuning
model.compile(optimizer=Adam(learning_rate=1e-5), loss="categorical_crossentropy", metrics=["accuracy"])

#  Fine-tune Model
history_fine = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test),
    class_weight=class_weights,
    callbacks=callbacks
)

#  Prediction Function
def predict_shade(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = pad_and_resize(img)
    img = np.expand_dims(img, axis=0)  # Add batch dimension
    prediction = model.predict(img)
    predicted_class = class_names[np.argmax(prediction)]
    print(f"Predicted Shade: {predicted_class}")
    return predicted_class

# Test example Prediction
test_image_path = "/content/drive/MyDrive/Dataset/3M_2/5.jpg"
predicted_shade = predict_shade(test_image_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Epoch 1/25
14/14 ━━━━━━━━━━━━━━━━━━━━ 37s 2s/step - accuracy: 0.0805 - loss: 3.4574 - val_accuracy: 0.1560 - val_loss: 2.7725 - learning_rate: 1.0000e-04
Epoch 2/25
14/14 ━━━━━━━━━━━━━━━━━━━━ 36s 2s/step - accuracy: 0.0938 - loss: 3.3729 - val_accuracy: 0.1560 - val_loss: 2.7275 - learning_rate: 1.0000e-04
Epoch 3/25
14/14 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.1255 - loss: 2.9191 - val_accuracy: 0.1743 - val_loss: 2.6774 - learning_rate: 1.0000e-04
Epoch 4/25
14/14 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - accuracy: 0.1434 - loss: 2.7222 - val_accuracy: 0.1927 - val_loss: 2.6258 - learning_rate: 1.0000e-04
Epoch 5/25
14/14 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step - accuracy: 0.1882 - loss: 2.4984 - val_accuracy: 0.2202 - val_loss: 2.5728 - learning_rate: 1.0000e-04
Epoch 6/25
14/14 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.2039 - loss: 2.1637 - val_accuracy: 0

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
Predicted Shade: 2R_1.5
